# 🏠 PanoWorld GPU 推理 - Colab 专用版

本地管理代码 → Colab 运行 GPU 推理

**配置来源**: 本地配置文件自动生成
**更新方式**: 在本地修改配置后重新运行 `sync_to_colab.py`

## 1️⃣ 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 配置路径（根据你的 Google Drive 实际路径修改）
DRIVE_MOUNT = '/content/drive/MyDrive'  # 或你的自定义路径
PROJECT_NAME = 'PanoWorld'
PROJECT_PATH = f"{DRIVE_MOUNT}/{PROJECT_NAME}"

import os
os.makedirs(PROJECT_PATH, exist_ok=True)
print(f"项目路径: {PROJECT_PATH}")

## 2️⃣ 克隆仓库（首次运行）或更新代码

In [ ]:
# 方式1: 从 GitHub 克隆（推荐首次使用）
# !cd {PROJECT_PATH} && git clone https://github.com/jjrCN/PanoWorld.git

# 方式2: 从 Google Drive 同步（推荐日常开发）
# 确保你已通过 sync_to_colab.py 将代码上传到 Drive

# 方式3: 直接上传代码到 Colab 运行时（小修改时）
print("请根据你的同步方式选择上方注释中的命令")

## 3️⃣ 安装依赖

In [ ]:
%cd {PROJECT_PATH}/PanoWorld

# 安装依赖
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q omegaconf easydict

print("✅ 依赖安装完成")

## 4️⃣ 检查 GPU

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 5️⃣ 下载模型权重（如未同步到 Drive）

In [ ]:
import os

os.makedirs("model_ckpt", exist_ok=True)

# 下载 1024x512 模型
!wget -q --show-progress \
    "https://huggingface.co/JiaJinrang/PanoWorld/resolve/main/model_ckpt/ckpt_panoworld_lrm_1024_512.pt" \
    -O model_ckpt/ckpt_panoworld_lrm_1024_512.pt

# 如需 2048x1024，取消下方注释
# !wget -q --show-progress \
#     "https://huggingface.co/JiaJinrang/PanoWorld/resolve/main/model_ckpt/ckpt_panoworld_lrm_2048_1024.ckpt" \
#     -O model_ckpt/ckpt_panoworld_lrm_2048_1024.ckpt

print("✅ 模型下载完成")

## 6️⃣ 配置推理参数

In [ ]:
# 根据你的实际数据路径修改
config_content = '''
model:
  class_name: model.PanoWorldLRM
  patch_factor: 2
  dim1: 256
  dim2: 512
  dim3: 1024
  num_register_tokens: 4
  head_dim: 64
  inter_multi: 4
  qk_norm: true
  stage1_nlayer: 2
  stage2_nlayer: 4
  stage3_nlayer: 8
  patch_size: 2
  in_channels: 12
  output_gs: true
  gaussians:
    sh_degree: 1
    opacity_degree: 2
    near_plane: 0.01
    far_plane: 1000000.0
    scale_bias: -6.9
    scale_max: -1.2
    opacity_bias: 1.0
    align_to_pixel: true
    max_dist: 15.0
    min_dist: 0.1
data:
  root_data_dir: "/content/drive/MyDrive/PanoWorld/data/RealSee3D_eval"
  data_path: "data_realsee3D/realsee3D_1024_1024_20260507_whole_room_map_json_eval_8views.txt"
  square_crop: true
  resize_h: 512
  resize_h_pano: 512
  sample_target_images: 6
  viewpoint_max_view: 8
  filter_top_down: false
training:
  train_stage: 1
inference:
  dataset_name: dataset.Dataset
  amp_dtype: bf16
  use_amp: true
  batch_size_per_gpu: 1
  num_threads: 8
  num_workers: 2
  prefetch_factor: 4
  use_tf32: true
  ckpt_path: model_ckpt/ckpt_panoworld_lrm_1024_512.pt
  out_dir: ./outputs/colab_inference
'''

with open("configs/colab_inference.yaml", "w") as f:
    f.write(config_content)

print("✅ 配置已生成: configs/colab_inference.yaml")

## 7️⃣ 运行推理

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python inference.py --config configs/colab_inference.yaml

## 8️⃣ 结果同步回 Google Drive（可选）

In [ ]:
import shutil

# 将结果复制到 Drive
shutil.copytree("./outputs/colab_inference", f"{DRIVE_MOUNT}/PanoWorld/outputs", dirs_exist_ok=True)
print("✅ 结果已同步到 Google Drive")